In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import time
import numpy as np
from loguru import logger
import warnings
warnings.filterwarnings("ignore")

#data loading
csv_basic = '../../Data/important/basic_eth_100d_scam.csv'
csv_scam = '../../Data/important/info_eth_100dscam.csv'
csv_scamliq = '../../Data/important/liq_eth_100d_scam.csv'
csv_rugpullseller = '../../Data/important/analysis/rugpull_sellers.csv'
csv_unique_wallet = '../../Data/important/analysis/unique_wallets.csv'


df_scaminfo = pd.read_csv(csv_scam)
df_scamliq = pd.read_csv(csv_scamliq)
df_rugpullseller = pd.read_csv(csv_rugpullseller)
df_unique_wallet = pd.read_csv(csv_unique_wallet)
df_single_seller = pd.read_csv("../../Data/important/analysis/single_sellerinfo.csv")
df_multi_seller = pd.read_csv("../../Data/important/analysis/multiseller_info.csv")


pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

print('Done data loading')

Done data loading


In [44]:
df_scaminfo 

,token_created_time,pool_created_time,pool_address,token_paired_address,verified_decimals,token_address,name,unverified_symbol,token_owner,id,fail
0,2018-04-16 00:04:52.000000,2018-11-02 22:56:11,0x4E0E28d426caf318747B8E05C8B0564A580E39a7,0x0000000000000000000000000000000000000000,18.0,0x919D0131fA5F77D99FBBBBaCe50bCb6E62332bf2,BorisCoin,BORIS,0xAd850d65eB5202f828f5f7883bc0B46ac87e64D4,3,False
1,2016-10-24 09:43:52.000000,2018-11-10 03:32:20,0x467fB51D54d7e51eE925F7f1a81AD5f2a0211169,0x0000000000000000000000000000000000000000,18.0,0x888666CA69E0f178DED6D75b5726Cee99A87D698,ICONOMI,ICN,0x00033390560d00f372ae50cD070b8124be5ecE5d,9,False
2,2017-10-19 20:27:35.000000,2018-12-09 19:41:39,0xC3c028721F854BC75967Cbe432fB0e221908Baa1,0x0000000000000000000000000000000000000000,18.0,0x9e88613418cF03dCa54D6a2cf6Ad934A78C7A17A,Swarm Fund Token,SWM,0x0ad59C344359Fdf8472E7FFbf4eB6AF4751138DA,30,False
3,2018-07-12 21:21:00.000000,2018-12-25 08:34:25,0x68326300DF49ec6387E75690857424c2ae111750,0x0000000000000000000000000000000000000000,18.0,0x737fA0372c8D001904Ae6aCAf0552d4015F9c947,MEDIBIT,MEDIBIT,0x5c4c0B176825311452764A2e7327C6b0273b2db2,47,False
4,2018-02-20 09:31:36.000000,2019-02-11 16:24:49,0x5d40522c20326F2Ebcec2D371f250e352E3BED27,0x0000000000000000000000000000000000000000,18.0,0xD49ff13661451313cA1553fd6954BD1d9b6E02b9,ElectrifyAsia,ELEC,0x0aC694A1b86645f2c8563E5fB66a6a22152a694f,79,False
...,...,...,...,...,...,...,...,...,...,...,...
105429,2024-07-08T07:32:59Z,2024-11-30T22:27:59Z,0x30600e7a89405e5bD76Be94dABB54d776F93E726,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,18.0,0x41B6161Fee8dc3AA822b5CBC0ee89e92Ab997a46,SnakeDice,SNAKE,0xEC055397730484b73a9308d24A2A365c86182804,387327,False
105430,2024-11-30T14:20:59Z,2024-11-30T23:01:11Z,0x6555fD6e6a1f048030314a26c5BFdfb42c9D3512,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,18.0,0x855F652bcA1AEa2D62A422620333162A3bbEbCa0,TOP HAT,HAT,0xf3c739AC6db9258D6152CBA121a96Fb9F5B7Efd8,387331,False
105431,2024-11-30T21:27:11Z,2024-11-30T23:01:47Z,0xa89f5Fefa693418d6A40ef68366493BaAA5c8212,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,18.0,0x3f7dB133aFf2F012C8534b36aB9731fe9Ee7bd43,Monerochan,MONEROCHAN,0x22748a8FfCC62940cbAb2f800E39832E16088C6E,387333,False
105432,2024-11-30T23:05:59Z,2024-11-30T23:05:59Z,0x9474F4C5b7a491cf223c960Cf52D60B147acBE17,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,18.0,0x749e4b7b6d92C57c9E910D61b4B434d770cAF2B5,PEPE GRINCH,PGRINCH,0x0E971EFD97F74054c5E1EBf2cA613ecCa5C7bF23,387335,False


In [ ]:
import ast
from collections import Counter

def parse_sellers(x):
    if pd.isna(x):
        return []
    if isinstance(x, (list, tuple, set)):
        return list(x)
    try:
        return ast.literal_eval(x)
    except Exception:
        return []

# count presence per row (each wallet counted at most once per row)
sellers_sets = df_rugpullseller['rugpull_sellers'].apply(lambda s: set(parse_sellers(s)))
cnt = Counter()
for s in sellers_sets:
    cnt.update(s)

# wallets that appear in more than one row
repeated_wallets = (
    pd.DataFrame(cnt.items(), columns=['wallet', 'row_count'])
      .sort_values('row_count', ascending=False)
      .query('row_count > 1')
      .reset_index(drop=True)
)

repeated_wallets.head(20)

,wallet,row_count
0,0xae2Fc483527B8EF99EB5D9B44875F005ba1FaE13,16525
1,0x77ad3a15b78101883AF36aD4A875e17c86AC65d1,8097
2,0x3fC91A3afd70395Cd496C647d5a6CC9D4B2b7FAD,3303
3,0xE592427A0AEce92De3Edee1F18E0157C05861564,2798
4,0xf9cAFEb32467994e3AFfd61E30865E5Ab32ABE68,2736
...,...,...
127247,0xf96FB6e3373a486C83ceB54E26e3B5a3201B3362,2
127248,0xa1f460688eB7A99Ce57F2CB3b67E2D0dA3A15404,2
127249,0xE72A1BD12a3Fe6F45F1A357aeEb3b8849711Fd6C,2
127250,0x871c7D3d310a3882037049fD3AdBBc731E9B1637,2


In [37]:
repeated_wallets.head(5)['wallet'].tolist()



['0xae2Fc483527B8EF99EB5D9B44875F005ba1FaE13',
 '0x77ad3a15b78101883AF36aD4A875e17c86AC65d1',
 '0x3fC91A3afd70395Cd496C647d5a6CC9D4B2b7FAD',
 '0xE592427A0AEce92De3Edee1F18E0157C05861564',
 '0xf9cAFEb32467994e3AFfd61E30865E5Ab32ABE68']

In [38]:
for i in repeated_wallets.head(5)['wallet'].tolist():
    wallet0 = i
    mask = (
        df_scamliq['sender_address'].fillna('').str.lower() == wallet0.lower()
    ) & (
        df_scamliq['token_address'].fillna('').str.lower() == '0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2'.lower()
    )
    # mask = mask[mask['token_address'] == '0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2']
    df_sender_wallet0 = df_scamliq[mask].copy()
    df_sender_wallet0['trade_value'] = df_sender_wallet0['amount_token'].fillna(0) * df_sender_wallet0['token_price'].fillna(0)
    total_value = df_sender_wallet0['trade_value'].sum()
    print(i,-total_value)

0xae2Fc483527B8EF99EB5D9B44875F005ba1FaE13 9009675.342213701
0x77ad3a15b78101883AF36aD4A875e17c86AC65d1 1276803.2250994365
0x3fC91A3afd70395Cd496C647d5a6CC9D4B2b7FAD 79378720.17696662
0xE592427A0AEce92De3Edee1F18E0157C05861564 59945861.34177524
0xf9cAFEb32467994e3AFfd61E30865E5Ab32ABE68 466592.4513535956


In [40]:
# keep using the previously computed `sellers_sets` and `parse_sellers`
mask_single = sellers_sets.apply(lambda s: len(s) == 1)

df_single_seller = df_rugpullseller[mask_single].copy()
df_single_seller['seller_list'] = df_single_seller['rugpull_sellers'].apply(parse_sellers)
df_single_seller['seller'] = df_single_seller['seller_list'].str[0]

# create dataframe for pools with multiple sellers (the rest)
mask_multi = ~mask_single
df_multi_seller = df_rugpullseller[mask_multi].copy()
df_multi_seller['seller_list'] = df_multi_seller['rugpull_sellers'].apply(parse_sellers)

print(f"rows with exactly one seller: {len(df_single_seller)}")
print(f"rows with multiple sellers: {len(df_multi_seller)}")
df_multi_seller.head()

rows with exactly one seller: 29573
rows with multiple sellers: 75861


,pool_address,rugpull_sellers,seller_count,pool_created_time,created_year,token_owner,seller_list
1,0x467fB51D54d7e51eE925F7f1a81AD5f2a0211169,"[""0x17559a8138E72C6Ef34512b11685dedC509BaDcf"",...",2,2018-11-10 03:32:20,2018,0x00033390560d00f372ae50cD070b8124be5ecE5d,"[0x17559a8138E72C6Ef34512b11685dedC509BaDcf, 0..."
2,0xC3c028721F854BC75967Cbe432fB0e221908Baa1,"[""0x006004fFA18E3cf78fA3b50393ec44C1ab89cF6c"",...",2,2018-12-09 19:41:39,2018,0x0ad59C344359Fdf8472E7FFbf4eB6AF4751138DA,"[0x006004fFA18E3cf78fA3b50393ec44C1ab89cF6c, 0..."
3,0x68326300DF49ec6387E75690857424c2ae111750,"[""0x52DA601C635951a464DA1E38503b7e9C04d7D530"",...",2,2018-12-25 08:34:25,2018,0x5c4c0B176825311452764A2e7327C6b0273b2db2,"[0x52DA601C635951a464DA1E38503b7e9C04d7D530, 0..."
4,0x5d40522c20326F2Ebcec2D371f250e352E3BED27,"[""0x35EfCe5f4bd52356e8215BCB9dC7687aDe6B6400"",...",7,2019-02-11 16:24:49,2019,0x0aC694A1b86645f2c8563E5fB66a6a22152a694f,"[0x35EfCe5f4bd52356e8215BCB9dC7687aDe6B6400, 0..."
8,0xCa265A7F4C9dc47b259850B696eBeFFA8BB18d9D,"[""0x48143eC3AcA8A255496e8ee7997591D3335Abf5f"",...",2,2019-05-18 19:59:37,2019,0x00806578B4B54D41224cfb1568eFB331DD74e25f,"[0x48143eC3AcA8A255496e8ee7997591D3335Abf5f, 0..."


In [42]:
df_multi_seller

,pool_address,rugpull_sellers,seller_count,pool_created_time,created_year,token_owner,seller_list
1,0x467fB51D54d7e51eE925F7f1a81AD5f2a0211169,"[""0x17559a8138E72C6Ef34512b11685dedC509BaDcf"",...",2,2018-11-10 03:32:20,2018,0x00033390560d00f372ae50cD070b8124be5ecE5d,"[0x17559a8138E72C6Ef34512b11685dedC509BaDcf, 0..."
2,0xC3c028721F854BC75967Cbe432fB0e221908Baa1,"[""0x006004fFA18E3cf78fA3b50393ec44C1ab89cF6c"",...",2,2018-12-09 19:41:39,2018,0x0ad59C344359Fdf8472E7FFbf4eB6AF4751138DA,"[0x006004fFA18E3cf78fA3b50393ec44C1ab89cF6c, 0..."
3,0x68326300DF49ec6387E75690857424c2ae111750,"[""0x52DA601C635951a464DA1E38503b7e9C04d7D530"",...",2,2018-12-25 08:34:25,2018,0x5c4c0B176825311452764A2e7327C6b0273b2db2,"[0x52DA601C635951a464DA1E38503b7e9C04d7D530, 0..."
4,0x5d40522c20326F2Ebcec2D371f250e352E3BED27,"[""0x35EfCe5f4bd52356e8215BCB9dC7687aDe6B6400"",...",7,2019-02-11 16:24:49,2019,0x0aC694A1b86645f2c8563E5fB66a6a22152a694f,"[0x35EfCe5f4bd52356e8215BCB9dC7687aDe6B6400, 0..."
8,0xCa265A7F4C9dc47b259850B696eBeFFA8BB18d9D,"[""0x48143eC3AcA8A255496e8ee7997591D3335Abf5f"",...",2,2019-05-18 19:59:37,2019,0x00806578B4B54D41224cfb1568eFB331DD74e25f,"[0x48143eC3AcA8A255496e8ee7997591D3335Abf5f, 0..."
...,...,...,...,...,...,...,...
105425,0x80777279644b44cBA3F5EC23B4AACDf76b106793,"[""0x111527f1386c6725a2F5986230f3060BDCAc041F"",...",45,2024-11-30T19:08:59Z,2024,0x55bD4fe4acd12F37bb05Db2bc87Bcfd8D947e15B,"[0x111527f1386c6725a2F5986230f3060BDCAc041F, 0..."
105426,0x6fFd814BBDd0c7D0Ab4050030f2536EC7AE6B0Ba,"[""0x00000000009E50a7dDb7a7B0e2ee6604fd120E49"",...",30,2024-11-30T20:20:59Z,2024,0x9560633fbAC92A84328205E22a6D0cFd0b790296,"[0x00000000009E50a7dDb7a7B0e2ee6604fd120E49, 0..."
105431,0xa89f5Fefa693418d6A40ef68366493BaAA5c8212,"[""0x000000d40B595B94918a28b27d1e2C66F43A51d3"",...",50,2024-11-30T23:01:47Z,2024,0x22748a8FfCC62940cbAb2f800E39832E16088C6E,"[0x000000d40B595B94918a28b27d1e2C66F43A51d3, 0..."
105432,0x9474F4C5b7a491cf223c960Cf52D60B147acBE17,"[""0x018f8E759a6F9Eb859F88f89De69F71B93379409"",...",21,2024-11-30T23:05:59Z,2024,0x0E971EFD97F74054c5E1EBf2cA613ecCa5C7bF23,"[0x018f8E759a6F9Eb859F88f89De69F71B93379409, 0..."


In [61]:
# count rows per tag in the `tmp` dataframe
tag_counts = tmp2['tag'].value_counts().reset_index()
tag_counts.columns = ['tag', 'count']
tag_counts


,tag,count
0,multi_nonowner,52547
1,multi-owner,23314


In [ ]:
#calculate tmp and tmp2: first-sell (each), sellcount (each), profit (each), sell-span (each, harder, take out if can)

df_single_seller
# df_multi_seller

,pool_address,rugpull_sellers,seller_count,pool_created_time,created_year,token_owner_x,seller_list,seller,tag,token_owner_y
0,0x4E0E28d426caf318747B8E05C8B0564A580E39a7,"[""0xA8C7372dC993d7510C9c45425807d463967cbb12""]",1,2018-11-02 22:56:11,2018,0xAd850d65eB5202f828f5f7883bc0B46ac87e64D4,['0xA8C7372dC993d7510C9c45425807d463967cbb12'],0xA8C7372dC993d7510C9c45425807d463967cbb12,single-nonowner,0xAd850d65eB5202f828f5f7883bc0B46ac87e64D4
1,0xbaf5A8BDF81cfE2d34c0CeD89236FE473183F2E8,"[""0x8948E4B00DEB0a5ADb909F4DC5789d20D0851D71""]",1,2019-03-07 05:59:20,2019,0x8948E4B00DEB0a5ADb909F4DC5789d20D0851D71,['0x8948E4B00DEB0a5ADb909F4DC5789d20D0851D71'],0x8948E4B00DEB0a5ADb909F4DC5789d20D0851D71,single-owner,0x8948E4B00DEB0a5ADb909F4DC5789d20D0851D71
2,0x225026D626E45FA662e6a71F679efF0CAc3054f1,"[""0x006004fFA18E3cf78fA3b50393ec44C1ab89cF6c""]",1,2019-03-11 14:01:50,2019,0xf1fa9a38914E853DE933FbF7Df2f278701e873DF,['0x006004fFA18E3cf78fA3b50393ec44C1ab89cF6c'],0x006004fFA18E3cf78fA3b50393ec44C1ab89cF6c,single-nonowner,0xf1fa9a38914E853DE933FbF7Df2f278701e873DF
3,0x9394C20adca4512DfC3d3c184c648E4193462Ebb,"[""0x2523C15dB0e3843DfC7C08772c8331Fb40CC8a0F""]",1,2019-04-22 08:02:06,2019,0x2523C15dB0e3843DfC7C08772c8331Fb40CC8a0F,['0x2523C15dB0e3843DfC7C08772c8331Fb40CC8a0F'],0x2523C15dB0e3843DfC7C08772c8331Fb40CC8a0F,single-owner,0x2523C15dB0e3843DfC7C08772c8331Fb40CC8a0F
4,0xEda88dDb13888C9A4dE7304965E9315E69ea980E,"[""0x866cb16F2162c319E48351827A7e15DDf0405E16""]",1,2019-06-10 00:06:22,2019,0x866cb16F2162c319E48351827A7e15DDf0405E16,['0x866cb16F2162c319E48351827A7e15DDf0405E16'],0x866cb16F2162c319E48351827A7e15DDf0405E16,single-owner,0x866cb16F2162c319E48351827A7e15DDf0405E16
...,...,...,...,...,...,...,...,...,...,...
29568,0x031D046Cbd8727D15702ECd5299eCcd63ADE84E8,"[""0xC14664811a2a4c233d253fDD03dee4B97ABBEbb5""]",1,2024-11-30T13:18:47Z,2024,0xCC54E2644ABb3B02EDb21aCD8F4f9ee06432Cada,['0xC14664811a2a4c233d253fDD03dee4B97ABBEbb5'],0xC14664811a2a4c233d253fDD03dee4B97ABBEbb5,single-nonowner,0xCC54E2644ABb3B02EDb21aCD8F4f9ee06432Cada
29569,0xd98d929EBc79856BBaEB783561294999c424211B,"[""0x670C625612D1662c91dc7B4434207101e7bd941f""]",1,2024-11-30T20:59:23Z,2024,0x670C625612D1662c91dc7B4434207101e7bd941f,['0x670C625612D1662c91dc7B4434207101e7bd941f'],0x670C625612D1662c91dc7B4434207101e7bd941f,single-owner,0x670C625612D1662c91dc7B4434207101e7bd941f
29570,0x6E7FE3428816cbD1CB1788aa84d04C4a08248b02,"[""0x670C625612D1662c91dc7B4434207101e7bd941f""]",1,2024-11-30T21:41:47Z,2024,0x670C625612D1662c91dc7B4434207101e7bd941f,['0x670C625612D1662c91dc7B4434207101e7bd941f'],0x670C625612D1662c91dc7B4434207101e7bd941f,single-owner,0x670C625612D1662c91dc7B4434207101e7bd941f
29571,0x30600e7a89405e5bD76Be94dABB54d776F93E726,"[""0x3328F7f4A1D1C57c35df56bBf0c9dCAFCA309C49""]",1,2024-11-30T22:27:59Z,2024,0xEC055397730484b73a9308d24A2A365c86182804,['0x3328F7f4A1D1C57c35df56bBf0c9dCAFCA309C49'],0x3328F7f4A1D1C57c35df56bBf0c9dCAFCA309C49,single-nonowner,0xEC055397730484b73a9308d24A2A365c86182804


In [ ]:
# build helper columns on df_scamliq (uses existing pd)
df_scamliq['sender_lc'] = df_scamliq['sender_address'].fillna('').str.lower()
df_scamliq['timestamp_dt'] = pd.to_datetime(df_scamliq['timestamp'], errors='coerce')

# get earliest sell timestamp per sender (vectorized)
sells = df_scamliq[df_scamliq['category'] == 'sell']
first_sell_map = sells.groupby('sender_lc')['timestamp_dt'].min().to_dict()

# loop through df_single_seller rows and populate "first sell time"
df_single_seller['first sell time'] = pd.NaT
for idx, row in df_single_seller.iterrows():
    seller = row.get('seller', '')
    if pd.isna(seller) or seller == '':
        continue
    ts = first_sell_map.get(str(seller).lower())
    if ts is not None:
        df_single_seller.at[idx, 'first sell time'] = ts
